# 15-01. 바이트 페어 인코딩 (Byte Pair Encoding, BPE)

기계에게 아무리 많은 단어를 학습시켜도 세상의 모든 단어를 알려줄 수는 없다. 학습하지 않은 단어가 등장하면 그 토큰을 **`UNK`(Unknown Token)**로 처리하는데, 이렇게 모르는 단어 때문에 문제 풀이가 어려워지는 상황을 **OOV(Out-Of-Vocabulary) 문제**라고 한다.

**서브워드 분리(subword segmentation)**는 하나의 단어를 더 작은 의미 단위의 서브워드들(예: `birthplace = birth + place`)로 나누어 인코딩·임베딩하려는 전처리 작업이다. 단어를 통째로 다루는 대신 서브워드 조합으로 표현하면 **OOV·희귀 단어·신조어** 문제를 크게 완화할 수 있다. 이 장에서는 대표적인 서브워드 분리 알고리즘인 **BPE(Byte Pair Encoding)**를 개념부터 코드까지 다룬다.

> **이 장의 흐름** : BPE의 기원(데이터 압축) → 자연어 처리에서의 BPE(글자 단위에서 단어 집합을 만드는 Bottom-up 접근) → 손으로 따라가는 예시(`low, lower, newest, widest`) → 코드 실습(`get_stats` → `merge_dictionary` → 병합 반복 → `bpe_codes`) → OOV 대처(`encode`로 미등록 단어 분리) → WordPiece·Unigram 토크나이저 개요.

> 실습 코드는 Sennrich et al.(2016) 논문에서 공개한 코드를 참고해 수정한 것으로, **외부 데이터 다운로드 없이** 예제 딕셔너리만으로 그 자리에서 전 과정을 실행할 수 있다.

# 1. BPE(Byte Pair Encoding)란?

**BPE**는 1994년에 제안된 **데이터 압축 알고리즘**이다. 이후 자연어 처리의 서브워드 분리에 응용되었는데, 먼저 원래의 압축 알고리즘이 어떻게 동작하는지 이해해 본다. BPE는 기본적으로 **연속으로 가장 많이 등장한 글자 쌍(byte pair)을 찾아 하나의 글자로 병합**하는 과정을 반복한다. 태생이 압축 알고리즘이라 여기서는 글자를 **바이트(byte)**라고 부른다.

다음 문자열에 BPE를 수행한다고 하자.

```
aaabdaaabac
```

가장 자주 등장하는 바이트 쌍은 `aa`이다. 이를 `Z`로 치환한다.

```
ZabdZabac          (Z = aa)
```

이제 가장 많이 등장하는 쌍은 `ab`이다. 이를 `Y`로 치환한다.

```
ZYdZYac            (Y = ab, Z = aa)
```

다시 가장 많은 쌍은 `ZY`이다. 이를 `X`로 치환한다.

```
XdXac              (X = ZY, Y = ab, Z = aa)
```

더 이상 병합할 쌍이 없으므로 BPE는 종료된다. 이처럼 **빈도가 높은 쌍을 반복적으로 병합**하는 것이 BPE의 핵심 아이디어다.

# 2. 자연어 처리에서의 BPE

- 논문 : https://arxiv.org/pdf/1508.07909.pdf

자연어 처리에서 BPE는 **서브워드 분리 알고리즘**으로, 기존 단어를 더 작은 단위로 분리한다는 의미다. 요약하면 **글자(character) 단위에서 시작해 점차 단어 집합(vocabulary)을 키워 나가는 Bottom-up 방식**이다. 먼저 훈련 데이터의 모든 단어를 글자(또는 유니코드) 단위로 쪼개 초기 단어 집합을 만들고, **가장 자주 등장하는 쌍을 하나로 통합**하는 과정을 정해진 횟수만큼 반복한다.

## 2-1. 기존의 접근

어떤 훈련 데이터에서 각 단어의 빈도수를 센 결과를 딕셔너리라고 하자.

```
# 훈련 데이터의 단어와 등장 빈도
low : 5,   lower : 2,   newest : 6,   widest : 3
```

여기서 단어 집합(중복 없는 단어들의 집합)은 `low, lower, newest, widest` 4개다. 이 경우 테스트 과정에서 **`lowest`**라는 단어가 등장하면, 기계는 이 단어를 학습한 적이 없으므로 제대로 대응하지 못하는 **OOV 문제**가 발생한다. 그렇다면 BPE를 적용하면 어떻게 달라질까?

## 2-2. BPE 알고리즘을 사용한 경우

먼저 딕셔너리의 모든 단어를 글자 단위로 분리한다. 이때부터 딕셔너리는 병합이 진행됨에 따라 계속 업데이트되며, 단어 집합을 갱신하기 위한 참고 자료 역할을 한다.

```
# dictionary (글자 단위로 분리)
l o w : 5,   l o w e r : 2,   n e w e s t : 6,   w i d e s t : 3

# vocabulary (초기 구성)
l, o, w, e, r, n, s, t, i, d
```

BPE의 특징은 **몇 회 반복(iteration)할지를 사용자가 정한다**는 점이다. 여기서는 10회로 가정한다. 가장 빈도가 높은 쌍을 하나로 통합하는 과정을 10회 반복한다.

- **1회** — 빈도 9로 가장 높은 `(e, s)` → `es` 로 통합
- **2회** — 빈도 9로 가장 높은 `(es, t)` → `est` 로 통합
- **3회** — 빈도 7로 가장 높은 `(l, o)` → `lo` 로 통합
- ... 이하 같은 방식으로 계속 통합

10회 반복 후 얻은 단어 집합은 다음과 같다.

```
l, o, w, e, r, n, s, t, i, d, es, est, lo, low, ne, new, newest, wi, wid, widest
```

이제 테스트 과정에서 `lowest`가 등장해도 더 이상 OOV가 아니다. 기계는 `lowest`를 우선 글자 단위(`l, o, w, e, s, t`)로 나눈 뒤, 단어 집합을 참고해 **`low`와 `est`** 두 서브워드로 인코딩한다. 두 서브워드 모두 단어 집합에 있으므로 OOV가 아니다.

> **핵심** : 단어 집합에 없는 단어라도, 그 단어를 구성하는 **서브워드들이 단어 집합에 있으면** 표현할 수 있다. BPE는 이렇게 OOV 문제를 완화한다.

## 2-3. 코드 실습하기

아래 코드는 원 논문에서 공개한 코드를 참고해 수정한 것이다. 우선 필요한 도구를 임포트하고, BPE를 몇 회 수행할지(`num_merges`)와 입력 딕셔너리를 정의한다. **`</w>`는 단어의 맨 끝에 붙이는 특수 문자**이며, 각 단어는 글자 단위로 분리해 둔다.

In [ ]:
import re, collections
from IPython.display import display, Markdown, Latex

# BPE를 몇 회 수행할지 지정 (여기서는 10회)
num_merges = 10

# BPE 입력 딕셔너리 : {글자 단위로 분리한 단어 : 빈도수}
# </w> 는 단어의 끝을 나타내는 특수 문자
dictionary = {
    'l o w </w>'      : 5,
    'l o w e r </w>'  : 2,
    'n e w e s t </w>': 6,
    'w i d e s t </w>': 3,
}

`get_stats`는 딕셔너리를 훑어 **인접한 두 유니그램(쌍)의 빈도수**를 집계한다. `merge_dictionary`는 가장 빈도가 높은 쌍을 받아, 딕셔너리 안에서 그 쌍을 하나로 병합한 새 딕셔너리를 돌려준다. 정규식의 `(?<!\S)`, `(?!\S)`는 병합 대상 쌍이 **다른 글자에 붙어 있지 않은, 정확히 공백으로 구분된 쌍**일 때만 치환하도록 하는 경계 조건이다.

In [ ]:
def get_stats(dictionary):
    # 유니그램 쌍(pair)들의 빈도수를 카운트한다.
    pairs = collections.defaultdict(int)
    for word, freq in dictionary.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq
    print('현재 pair들의 빈도수 :', dict(pairs))
    return pairs

def merge_dictionary(pair, v_in):
    # 가장 빈도가 높은 쌍(pair)을 딕셔너리 안에서 하나로 병합한다.
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')  # 공백으로 정확히 구분된 쌍만 매칭
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

이제 위 두 함수를 이용해 **가장 빈도가 높은 쌍을 하나의 유니그램으로 통합하는 과정을 `num_merges`회 반복**한다. 병합할 때마다 그 기록을 `bpe_codes`에 저장해 둔다. (이 기록은 뒤에서 새로운 단어를 분리할 때 참고한다.)

In [ ]:
bpe_codes = {}
bpe_codes_reverse = {}

for i in range(num_merges):
    display(Markdown("### Iteration {}".format(i + 1)))
    pairs = get_stats(dictionary)
    best = max(pairs, key=pairs.get)              # 빈도가 가장 높은 쌍
    dictionary = merge_dictionary(best, dictionary)
    bpe_codes[best] = i                           # 병합 순서를 기록
    bpe_codes_reverse[best[0] + best[1]] = best
    print("new merge: {}".format(best))
    print("dictionary: {}".format(dictionary))

`(e, s)` 쌍은 초기 딕셔너리에서 총 9회 등장해 가장 먼저 `es`로 통합된다. 이어서 `(es, t)`, `(est, </w>)` 순으로 병합된다. 이렇게 빈도가 높은 순서대로 통합하는 과정을 `num_merges`회 반복한 것이다.

`bpe_codes`를 출력하면 **어떤 쌍이 몇 번째로 병합되었는지** 기록을 볼 수 있다. 이 기록은 새로운 단어가 등장했을 때, 현재 서브워드 단어 집합을 근거로 그 단어를 분리하는 데 사용된다.

In [ ]:
print(bpe_codes)

## 2-4. OOV에 대처하기

이제 학습한 `bpe_codes`를 이용해 **처음 보는 단어를 서브워드로 분리**해 본다. `get_pairs`는 한 단어 안의 인접 심볼 쌍 집합을 구하고, `encode`는 그 쌍들 중 **`bpe_codes`에서 병합 우선순위가 가장 높은(= 먼저 학습된) 쌍**부터 순서대로 병합해 나간다. 더 이상 `bpe_codes`에 있는 쌍이 없으면 병합을 멈춘다.

In [ ]:
def get_pairs(word):
    """단어 안의 인접 심볼 쌍(pair) 집합을 반환한다.
    word는 (심볼, 심볼, ...) 형태의 튜플이며, 심볼은 가변 길이 문자열이다."""
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs

def encode(orig):
    """학습된 BPE 병합 규칙(bpe_codes)을 순서대로 적용해 단어를 인코딩한다."""
    word = tuple(orig) + ('</w>',)
    display(Markdown("__word split into characters:__ <tt>{}</tt>".format(word)))

    pairs = get_pairs(word)
    if not pairs:
        return orig

    iteration = 0
    while True:
        iteration += 1
        display(Markdown("__Iteration {}:__".format(iteration)))
        print("bigrams in the word: {}".format(pairs))

        # bpe_codes에서 병합 우선순위(= 학습 순서)가 가장 높은 쌍을 고른다.
        bigram = min(pairs, key=lambda pair: bpe_codes.get(pair, float('inf')))
        print("candidate for merging: {}".format(bigram))

        if bigram not in bpe_codes:
            display(Markdown("__Candidate not in BPE merges, algorithm stops.__"))
            break

        first, second = bigram
        new_word = []
        i = 0
        while i < len(word):
            try:
                j = word.index(first, i)
                new_word.extend(word[i:j])
                i = j
            except:
                new_word.extend(word[i:])
                break

            if word[i] == first and i < len(word) - 1 and word[i + 1] == second:
                new_word.append(first + second)   # 두 심볼을 하나로 병합
                i += 2
            else:
                new_word.append(word[i])
                i += 1

        new_word = tuple(new_word)
        word = new_word
        print("word after merging: {}".format(word))

        if len(word) == 1:
            break
        else:
            pairs = get_pairs(word)

    # 특수 토큰 </w> 는 출력하지 않는다.
    if word[-1] == '</w>':
        word = word[:-1]
    elif word[-1].endswith('</w>'):
        word = word[:-1] + (word[-1].replace('</w>', ''),)

    return word

단어 `loki`가 들어오면 BPE는 이 단어를 어떻게 분리할까? 현재 서브워드 단어 집합에 `lo`가 존재하므로 `lo`는 유지하고, 나머지 `k`, `i`는 분리된다.

In [ ]:
encode("loki")

`lowest`에 대해서도 수행해 본다. 서브워드 집합에 `low`와 `est`가 있으므로 **`low`, `est`**로 분리된다.

In [ ]:
encode("lowest")

`lowing`은 어떨까? `low`는 존재하지만 `i`, `n`, `g`로 이어지는 서브워드는 학습되지 않았으므로, `low`만 묶이고 나머지 `i`, `n`, `g`는 모두 분리된다.

In [ ]:
encode("lowing")

마지막으로, 훈련 데이터의 어떤 서브워드와도 겹치지 않는 `highing`을 넣어 본다. 병합할 수 있는 쌍이 없으므로 **모든 알파벳이 그대로 분리**된다.

In [ ]:
encode("highing")

# 3. WordPiece Tokenizer

- 논문 : https://static.googleusercontent.com/media/research.google.com/ko//pubs/archive/37842.pdf
- 구글 번역기 적용 논문 : https://arxiv.org/pdf/1609.08144.pdf

**WordPiece Tokenizer**는 BPE의 변형이다. BPE가 **빈도수**가 가장 높은 쌍을 병합하는 것과 달리, WordPiece는 **병합했을 때 코퍼스의 우도(Likelihood)를 가장 크게 높이는 쌍**을 병합한다. 2016년 논문에서 구글은 번역기에 WordPiece를 적용한 결과를 다음과 같이 소개했다.

- 수행 전 : `Jet makers feud over seat width with big orders at stake`
- 수행 후 : `_J et _makers _fe ud _over _seat _width _with _big _orders _at _stake`

`Jet`은 `J`와 `et`로, `feud`는 `fe`와 `ud`로 나뉜 것을 볼 수 있다. WordPiece는 **모든 단어의 맨 앞에 `_`를 붙이고** 서브워드를 통계 기반으로 띄어쓰기로 분리한다. 여기서 언더바 `_`는 문장 복원을 위한 장치다. 서브워드를 구분하려고 새로 추가된 띄어쓰기와, 원래 문장에 있던 띄어쓰기를 구별하기 위해서다. 복원할 때는 **현재의 모든 띄어쓰기를 제거한 뒤 `_`를 띄어쓰기로 바꾸면** 원문이 된다. 이 알고리즘은 **BERT**를 훈련할 때도 사용되었다.

# 4. Unigram Language Model Tokenizer

- 논문 : https://arxiv.org/pdf/1804.10959.pdf

**유니그램 언어 모델 토크나이저**는 각 서브워드마다 **손실(loss)**을 계산한다. 여기서 서브워드의 손실이란, 그 서브워드를 단어 집합에서 **제거했을 때 코퍼스의 우도(Likelihood)가 감소하는 정도**를 말한다. 이렇게 측정한 손실을 기준으로 서브워드를 정렬해, **가장 영향이 적은(최악의 영향을 주는) 10~20%의 토큰을 제거**한다. 이 과정을 원하는 단어 집합 크기에 도달할 때까지 반복한다.

---

이로써 **서브워드 토크나이저**의 대표 알고리즘인 **BPE**를 개념부터 코드까지 정리했다. BPE는 **글자 단위에서 시작해 빈도가 높은 쌍을 반복적으로 병합**하는 Bottom-up 방식으로 단어 집합을 만들고, 그 결과 단어 집합에 없는 단어라도 **서브워드들의 조합으로 표현**해 OOV 문제를 완화한다. 핵심 흐름은 (1) `get_stats`로 쌍의 빈도를 세고, (2) 가장 빈도가 높은 쌍을 `num_merges`회 병합하며, (3) 병합 기록(`bpe_codes`)을 근거로 새로운 단어를 서브워드로 분리한다는 것이다.

BPE 외에도 **병합 시 우도를 높이는 쌍**을 고르는 **WordPiece**(BERT에 사용), **손실이 적은 토큰을 제거**해 나가는 **Unigram Language Model** 등의 변형이 있다. 이어서 이 알고리즘들을 실무에서 사용하기 위한 패키지인 **센텐스피스(SentencePiece)**나 **토크나이저스(tokenizers)**의 사용법을 학습한다.